# Milestone 2: RAG Exploration

This notebook has the initial explorations of the RAG pipeline.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
import faiss
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from groq import Groq
import os
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

In [2]:
# -------------------------
# Load data
# -------------------------
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
META_FILE   = PROJECT_ROOT / "data" / "raw" / "meta_Toys_and_Games.jsonl"
REVIEW_FILE = PROJECT_ROOT / "data" / "raw" / "Toys_and_Games.jsonl"

meta = pd.read_json(META_FILE, lines=True, nrows=50000)
review = pd.read_json(REVIEW_FILE, lines=True, nrows=50000)

cleaned_meta = meta.drop(columns=['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'], errors='ignore')
cleaned_meta.head()

reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns=['images', 'timestamp', 'user_id', 'verified_purchase'], errors='ignore')
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [3]:
# -------------------------
# Clean text columns
# -------------------------
cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
).str.lower()

cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
).str.lower()

cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['title'] = cleaned_meta['title'].str.lower()

cleaned_meta = cleaned_meta[
    (cleaned_meta['title'].str.strip() != '') &
    (cleaned_meta['description'].str.strip() != '') &
    (cleaned_meta['features'].str.strip() != '') &
    (cleaned_meta['categories'].str.strip() != '')
].reset_index(drop=True)

In [4]:
# -------------------------
# Prepare review text per product
# -------------------------
cleaned_reviews = cleaned_reviews.copy()
review_text_cols = [col for col in ['title', 'text'] if col in cleaned_reviews.columns]
cleaned_reviews['combined_review_text'] = cleaned_reviews[review_text_cols].fillna('').agg(' '.join, axis=1)
cleaned_reviews['combined_review_text'] = cleaned_reviews['combined_review_text'].str.lower()

grouped_reviews = (
    cleaned_reviews.groupby('parent_asin')['combined_review_text']
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

rag_df = cleaned_meta.merge(grouped_reviews, on='parent_asin', how='left')
rag_df['combined_review_text'] = rag_df['combined_review_text'].fillna('')

In [5]:
# -------------------------
# Build product documents
# -------------------------
products = (
    rag_df['title'] + ' ' +
    rag_df['description'] + ' ' +
    rag_df['features'] + ' ' +
    rag_df['categories'] + ' ' +
    rag_df['combined_review_text']
).tolist()

In [6]:
# -------------------------
# Embeddings and vector store
# -------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# -------------------------
# Retriever
# -------------------------

def retrieve(query, top_k=5):
    """Retrieve the top-k most semantically similar products 
    from the RAG dataframe for a given query."""
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return rag_df.iloc[indices[0]]

# -------------------------
# Building context
# -------------------------
# Implemented with the help of chatGPT
def build_context(docs):
    """Format retrieved product rows into a structured text context string for the LLM."""
    blocks = []
    for _, row in docs.iterrows():
        review_snippet = row.get('combined_review_text', '')[:500]

        block = (
            f"Product ASIN: {row.get('parent_asin', 'N/A')}\n"
            f"Title: {row.get('title', '')}\n"
            f"Description: {row.get('description', '')}\n"
            f"Features: {row.get('features', '')}\n"
            f"Categories: {row.get('categories', '')}\n"
            f"Review Evidence: {review_snippet}\n"
        )
        blocks.append(block)
    return "\n\n".join(blocks)


# -------------------------
# Wrapper function
# -------------------------
def retrieve_and_build_context(query):
    """Retrieve the top-k relevant products and 
    return them as a formatted LLM context string."""
    docs = retrieve(query, top_k=5)
    return build_context(docs)

In [8]:
# -------------------------
# Prompt variants
# -------------------------
prompt1 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

prompt2 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Keep the answer shorter than 3 sentences.
- Make sure nothing is repeated, and only necessary details are written.
- If the answer is not in the context, say: "The context does not provide enough information."

Context:
{context}

Input:
{input}

Answer:
"""
)

prompt3 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Be clear and helpful
- Give specific statements instead of general ones. 
- If something is missing, say "not enough context to answer your question"

Context:
{context}

Question:
{input}

Answer:
"""
)

In [9]:
# please uncomment this line below and add your GROQ API key before proceeding
# import os
# os.environ["GROQ_API_KEY"] = "YOUR KEY HERE"

In [10]:
# -------------------------
# Rag Pipeline model 1
# -------------------------
# Implemented with the help of chatGPT

load_dotenv()

llm1 = ChatGroq(model="llama-3.1-8b-instant")

prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm1
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))

rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm1
    | StrOutputParser()
)

queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids",
    "Toys for learning colors",
    "A toy for sensory development"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


===== prompt1 =====
Based on the context, a suitable answer for a good board game for kids age 8 and up is the Sorry! board game for kids ages 6 and up (Product ASIN: B00000IWD0) and the Chess, Checkers & Tic-Tac-Toe with Dual Sided Gameboard (Product ASIN: B07P2VVFVM) is recommended for ages 6 and up.

However, considering the age requirement of 8 and up, the Sorry! board game would be a suitable choice as it's recommended for ages 6 and up.

For kids aged 8 and above, the Sorry! board game is a great choice as it's designed for that age group and it provides a fun way to teach kids about ploting their own revenge and developing strategic thinking.

===== prompt2 =====
The Sorry! board game for kids ages 6 and up is not suitable for kids under age 8 due to potential frustration and tears.

===== prompt3 =====
Based on the context, I see that the Sorry! board game is recommended for kids ages 6 and up, but it's mentioned that children under age 8 might find it frustrating. However, si

In [11]:
# -----------------------------------------
# Rag Pipeline model 2 for final submission
# -----------------------------------------

llm2 = ChatGroq(model="llama-3.3-70b-versatile")

prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm2
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))

rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm2
    | StrOutputParser()
)

queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids",
    "Toys for learning colors",
    "A toy for sensory development"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


===== prompt1 =====
The Sorry! board game is a good option for kids aged 6 and up, but considering the age of 8 and up, it's still suitable. Another option is the Chess, Checkers & Tic Tac Toe game, which is recommended for ages 6 and up.

===== prompt2 =====
The Sorry! board game is suitable for kids ages 6 and up, making it a good option for kids age 8 and up.

===== prompt3 =====
The "Sorry! Board Game" (Product ASIN: B00000IWD0) is a good option for kids aged 8 and up. This classic Hasbro board game is designed for kids aged 6 and up, but it's noted that children under age 8 may struggle with the frustration of having their pawns sent back to the start. At age 8 and up, kids can better handle this aspect of the game and develop strategies to plot their own revenge.

QUERY: A good board game for kids age 8 and up
The Sorry! board game is a good option for kids age 8 and up. It's a classic game of strategy, chance, and luck that's easy to grasp for children as young as 6 years old, 